# Bayesian Fundamentals

## Overview

Bayesian inference updates prior beliefs with observed data to produce a posterior distribution over parameters. Unlike frequentist inference, parameters are treated as random variables with probability distributions.

**Bayes' theorem:**

```
posterior ∝ likelihood × prior
P(θ | data) ∝ P(data | θ) × P(θ)
```

**Key concepts:**

| Concept | Meaning |
|---|---|
| Prior P(θ) | Belief about parameter before seeing data |
| Likelihood P(data \| θ) | Probability of data given parameters |
| Posterior P(θ \| data) | Updated belief after seeing data |
| Credible interval | Interval containing θ with specified posterior probability |
| MAP | Maximum a posteriori: mode of the posterior |

A 95% credible interval has a direct probability interpretation: there is a 95% posterior probability that θ lies in this interval.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import betaln

rng = np.random.default_rng(42)
# Ecological example: estimating restoration success rate
# True unknown probability that a restored site reaches target richness
true_p = 0.65
n_sites = 40
successes = rng.binomial(1, true_p, n_sites)
n_success = successes.sum()
print(f"Observed: {n_success}/{n_sites} sites successful ({n_success/n_sites:.2f})")
print(f"True p: {true_p}")

---
## Conjugate Prior: Beta-Binomial

In [ ]:
# Beta(a,b) is conjugate prior for Binomial likelihood
# Posterior: Beta(a + successes, b + failures)
theta = np.linspace(0, 1, 500)
failures = n_sites - n_success

priors = [
    (1,  1,  "Uniform Beta(1,1)"),
    (2,  2,  "Weakly informative Beta(2,2)"),
    (5,  3,  "Informative Beta(5,3) prior mean=0.625"),
]
fig, axes = plt.subplots(1,3,figsize=(14,4))
for ax, (a0,b0,label) in zip(axes, priors):
    prior_pdf     = stats.beta.pdf(theta, a0, b0)
    likelihood    = stats.beta.pdf(theta, n_success+1, failures+1)  # unnorm
    posterior_pdf = stats.beta.pdf(theta, a0+n_success, b0+failures)
    ax.plot(theta, prior_pdf,     color="grey",      lw=2, linestyle="--", label="Prior")
    ax.plot(theta, likelihood,    color="steelblue", lw=2, linestyle=":",  label="Likelihood")
    ax.plot(theta, posterior_pdf, color="#e74c3c",   lw=2.5, label="Posterior")
    ax.axvline(true_p, color="navy", lw=1.5, linestyle="--", alpha=0.6, label=f"True p={true_p}")
    a_post, b_post = a0+n_success, b0+failures
    ci = stats.beta.ppf([0.025, 0.975], a_post, b_post)
    ax.set_title(f"{label}\nPosterior mean={a_post/(a_post+b_post):.3f}, 95% CI [{ci[0]:.2f},{ci[1]:.2f}]")
    ax.set_xlabel("p (success probability)"); ax.legend(fontsize=7)
plt.suptitle("Beta-Binomial: Prior × Likelihood → Posterior", y=1.01)
plt.tight_layout(); plt.show()

---
## Prior Sensitivity

In [ ]:
# How much does the posterior change across different priors?
prior_configs = [(1,1),(2,2),(5,3),(10,5),(0.5,0.5)]
fig, ax = plt.subplots(figsize=(9,5))
colors = plt.cm.viridis(np.linspace(0.1,0.9,len(prior_configs)))
for (a0,b0), color in zip(prior_configs, colors):
    a_post = a0+n_success; b_post = b0+failures
    post_mean = a_post/(a_post+b_post)
    ci = stats.beta.ppf([0.025,0.975], a_post, b_post)
    ax.plot(theta, stats.beta.pdf(theta,a_post,b_post), color=color, lw=2,
            label=f"Beta({a0},{b0}): mean={post_mean:.3f} [{ci[0]:.2f},{ci[1]:.2f}]")
ax.axvline(n_success/n_sites, color="#e74c3c", lw=1.5, linestyle="--", label="MLE")
ax.axvline(true_p, color="navy", lw=1.5, linestyle=":", label=f"True p={true_p}")
ax.set_xlabel("p"); ax.set_ylabel("Posterior density")
ax.set_title(f"Prior Sensitivity (n={n_sites}): With enough data, posteriors converge")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

---
## Sequential Updating

In [ ]:
# Bayesian updating: each observation updates the posterior
a, b = 2, 2  # starting prior
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(theta, stats.beta.pdf(theta,a,b), color="grey", lw=1.5,
        linestyle="--", label="Prior Beta(2,2)", alpha=0.7)
colors_seq = plt.cm.Blues(np.linspace(0.3,0.9,5))
for i, n_obs in enumerate([1,5,10,20,40]):
    obs = successes[:n_obs]
    a_n = a + obs.sum()
    b_n = b + (n_obs - obs.sum())
    ax.plot(theta, stats.beta.pdf(theta,a_n,b_n), color=colors_seq[i], lw=2,
            label=f"After {n_obs} obs: mean={a_n/(a_n+b_n):.3f}")
ax.axvline(true_p, color="#e74c3c", lw=1.5, linestyle=":", label=f"True p={true_p}")
ax.set_xlabel("p"); ax.set_ylabel("Density")
ax.set_title("Sequential Bayesian Updating: Posterior Concentrates as Data Accumulates")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# Normal-Normal conjugate: estimating mean richness
# Prior: mu ~ N(mu0, sigma0^2), known sigma
mu0, sigma0 = 18.0, 5.0  # prior
sigma_known = 4.0         # known measurement SD
richness_obs = rng.normal(21.5, sigma_known, 30)
n_obs = len(richness_obs)
# Posterior: N(mu_post, sigma_post^2)
prec_prior = 1/sigma0**2
prec_like  = n_obs/sigma_known**2
sigma_post = np.sqrt(1/(prec_prior+prec_like))
mu_post    = sigma_post**2 * (mu0/sigma0**2 + richness_obs.mean()*n_obs/sigma_known**2)
print(f"Prior: N({mu0}, {sigma0}^2)")
print(f"Likelihood: {n_obs} obs, sample mean={richness_obs.mean():.2f}")
print(f"Posterior: N({mu_post:.2f}, {sigma_post:.2f}^2)")
ci = stats.norm.ppf([0.025,0.975], mu_post, sigma_post)
print(f"95% Credible interval: [{ci[0]:.2f}, {ci[1]:.2f}]")

---

## Common Pitfalls

**1. Confusing credible intervals with confidence intervals**  
A 95% Bayesian credible interval means there is 95% posterior probability that the parameter lies within it — a direct probability statement. A 95% frequentist confidence interval means 95% of such intervals constructed over repeated sampling would contain the true value. They answer different questions and should not be interpreted interchangeably.

**2. Treating uniform priors as "objective" or "uninformative"**  
A Uniform(0,1) prior on a probability is uniform on the probability scale but not on the log-odds scale. There is no universally uninformative prior. A weakly informative prior (e.g. Beta(2,2)) that gently regularises toward 0.5 is often more defensible than a flat prior.

**3. Not checking prior sensitivity**  
If the posterior changes substantially across reasonable prior choices, the data do not dominate the prior — the posterior is prior-sensitive and results depend heavily on the prior specification. Always report a prior sensitivity analysis.

**4. Ignoring the prior when n is small**  
With n=5 observations, the prior contributes substantially to the posterior. Reporting only the posterior mean without showing the prior and the amount of data gives a false impression of data-driven certainty.

**5. Assuming conjugacy is always available**  
Conjugate priors exist for simple models (Beta-Binomial, Normal-Normal, Gamma-Poisson) but not for most complex models. For hierarchical models, regression with non-standard likelihoods, or latent variable models, MCMC sampling (PyMC, Stan) is required.
---
*python_methods_library - Samantha McGarrigle*